# Lab 2: Exploring the Fund's Data

**Time:** about 90 minutes

You've just been handed a raw extract from the Fund's systems: employers, members, monthly contribution
remittances and benefit claims. Before anyone builds a model on it, someone has to understand it. What does
one row mean? What's missing, and why? What looks wrong? What looks *interesting*?

That someone is you. By the end of this notebook you'll have:

- loaded and typed all four tables properly,
- a written list of data-quality issues, each with a proposed fix,
- a handful of business questions worth taking to the claims, compliance and finance teams.

Work through the cells in order. Wherever you see **Your turn**, stop and write the code yourself before
moving on. Suggested answers are at the very end of the notebook, but give it a real try first.

**How the code is written.** Every code cell does one small step and starts with a comment saying what it does.
Read the comment, run the cell, look at the output, then move on. Nothing here is clever on purpose: it's the same
pandas you'd write at work.

## Where this lab fits

**What you've done so far.** In Lab 1 you built the workspace for the whole course: a Python environment called
`fund-ai`, a project folder, and Git to keep track of your changes. Nothing has touched the Fund's data yet.

**What you'll do now.** You'll open the four raw tables and get to know them the way an analyst does on their first
week in a new job: what each row means, which values can be trusted, which can't, and what the data says about how the
Fund works.

**Why this comes first.** Every model in the next nine days learns from data like this. A model can't tell a typing
error from a real fact. If a wage of TZS 80 million is really a slip for TZS 8 million, the model will happily learn
from it. The saying in the trade is *"garbage in, garbage out"*. Exploring the data is how you find the garbage before
a model eats it.

**What happens next.** In Lab 3 you'll take the problems you write down today and turn each one into a cleaning rule
in a pipeline that runs the same way every time. From Day 2 onwards, the models use the clean data that comes out of
that pipeline.

### How to read this notebook

Two kinds of text block are new in this version of the labs:

- **Theory** blocks explain an idea from scratch before you use it. Read them even if you've met the idea before: they
  say how it applies to the Fund's data.
- **Where we are** blocks are short summaries between sections: what you've found so far, what comes next, and why.
  If you lose the thread, read back through those.

## 1. Setup

Run the next cells. They import the libraries you need and find the data files. The files can sit next to this
notebook or in `../data/raw/` if you set up the project layout from Lab 1.

In [ ]:
# Import the libraries for this lab.
#   numpy  - fast arrays of numbers (pandas is built on it)
#   pandas - tables of data (DataFrames)
#   matplotlib - charts
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Display settings: show more columns, and print numbers with thousand separators and 2 decimals.
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", "{:,.2f}".format)

# Default chart size (width, height in inches).
plt.rcParams["figure.figsize"] = (9, 4)

In [ ]:
# Where are the CSV files?
# If you followed the Lab 1 layout they are in ../data/raw. Otherwise they are next to this notebook.
DATA = Path("../data/raw")
if not (DATA / "members.csv").exists():
    DATA = Path(".")

# The date the extract was taken. Any date after this is impossible.
SNAPSHOT = pd.Timestamp("2026-01-31")

print("Python:", sys.executable)
print("Reading data from:", DATA.resolve())

### Theory: how a computer holds a table

A spreadsheet looks like a grid of cells, and any cell can hold anything: a number, a word, a date, a formula. Python's
data tools work differently, and knowing how saves a lot of confusion later.

**Columns, not cells.** pandas stores a table (a **DataFrame**) as a set of columns. Each column is a **Series**: a
long list of values that all have the same **data type** (dtype). The main types you'll meet are:

| Type | Holds | Example |
|---|---|---|
| `int64` | whole numbers | number of members reported: `42` |
| `float64` | numbers with decimals, and the special "missing" value `NaN` | a wage: `650000.0` |
| `string` / `object` | text | a region: `"Arusha"` |
| `datetime64` | dates and times | a claim date: `2024-03-15` |
| `bool` | `True` / `False` | "was it paid on time?" |
| `category` | text from a short, fixed list, stored compactly | benefit type: `MATERNITY` |

**Why one type per column.** Because every value in a column has the same type, pandas can store the column as one
compact block of memory and run an operation on all of it at once. That's what makes pandas fast. It's also why one
bad value matters: a single `"N/A"` typed into a column of numbers forces the whole column to be stored as text, and
then you can't add it up.

**Underneath, it's numpy.** Each column is held in a **numpy array**. numpy is the library that does fast arithmetic on
long lists of numbers. You'll rarely call it yourself, but three of its habits (below) explain behaviour in pandas that
otherwise looks strange.

**The index.** Every DataFrame also has an **index**: a label for each row. When you read a CSV the index is just
`0, 1, 2, ...`. You'll mostly ignore it in this lab, but you'll see it on the left of every table pandas prints.

## 2. A few minutes of numpy

pandas is built on numpy arrays. You rarely call numpy directly, but three of its habits explain a lot of
pandas behaviour that otherwise seems strange. Run each cell and read the output before moving on.

**Vectorised arithmetic.** You never loop over rows to compute a contribution. The operation applies to the
whole array at once.

In [ ]:
# Four monthly wages, in TZS.
wages = np.array([450_000, 800_000, 1_250_000, 380_000])

# Multiply every wage by 20% in one go - no loop needed.
contribution = wages * 0.20
contribution

**dtypes matter.** An integer array can't hold a missing value. The moment a value is missing, numpy (and
pandas) switches to float and uses `NaN`. That's why ID numbers, or any very large number, read from a CSV can turn
into `1.04e+08` (scientific notation: 1.04 × 10⁸) if you're careless, and why you'll tell pandas to read IDs as text.

In [ ]:
# An array of whole numbers...
a = np.array([1, 2, 3])
# ...and the same array with one missing value (np.nan).
b = np.array([1, 2, np.nan])

# The "dtype" is the type of value the array holds. Notice b had to become float.
print("a:", a.dtype)
print("b:", b.dtype)

**NaN spreads.** Any arithmetic with NaN gives NaN, and `NaN == NaN` is `False`. Use `np.isnan` or pandas'
`isna()` to test for it, and the `nan`-aware functions when you want to ignore it.

In [ ]:
# A normal sum gives nan, because one value is missing.
print("sum:", b.sum())

# np.nansum skips the missing value.
print("nansum:", np.nansum(b))

In [ ]:
# NaN is not even equal to itself, so never test for it with ==
print("np.nan == np.nan ->", np.nan == np.nan)

# Use np.isnan (or .isna() in pandas) instead.
print("np.isnan(b) ->", np.isnan(b))

> **Where we are.** You've seen how pandas stores data and why types matter. Now you'll load the four real tables.

### Theory: CSV files, and how the four tables fit together

**A CSV file has no types.** "CSV" means *comma-separated values*. It's plain text: one line per row, with commas
between the values. Open one in Notepad and you'll see something like:

```
member_no,employer_no,date_of_birth,monthly_wage_tzs
00104233,E01120,1985-04-12,650000
```

Nothing in the file says that `00104233` is a label and not a number, or that `1985-04-12` is a date. When pandas reads
the file it **guesses** each column's type from what it sees. It guesses numbers well, and it gets labels wrong: it
reads `00104233` as the number 104233 and drops the leading zeros. That's why you'll tell it which columns are IDs.

**Four tables, linked by keys.** The Fund's data is split into four tables, the way most business systems store it.
Each table describes one kind of thing, and they point at each other through shared ID columns called **keys**:

```
employers ──(employer_no)──► members ──(member_no)──► claims
    │
    └──(employer_no)──► contributions (one row per employer per month)
```

- An **employer** has many **members** working for it.
- An employer sends one **contribution** remittance every month.
- A **member** can lodge several **claims** over their life.

Splitting the data like this avoids repeating information. The employer's sector is stored once in `employers`, not
copied onto every one of that employer's monthly contribution rows. When you need the two together, you **join** them on the key
(Section 11).

## 3. Load the four tables

Two things to notice in the next cells:

- ID columns (`member_no`, `employer_no`, `nida_no`, `phone`) are read as **text**. They're labels, not
  numbers. You'd never add two member numbers together, and reading them as numbers would drop leading
  zeros from phone numbers.
- Dates are **not** parsed yet. You'll do that deliberately in Section 6, so that bad dates show up
  instead of silently disappearing.

In [ ]:
# Tell pandas to read these ID columns as text ("string"), not as numbers.
id_cols = {
    "member_no": "string",
    "employer_no": "string",
    "nida_no": "string",
    "phone": "string",
    "claim_no": "string",
    "tin": "string",
}

In [ ]:
# Read each CSV file into a DataFrame (a table).
employers = pd.read_csv(DATA / "employers.csv", dtype=id_cols)
members = pd.read_csv(DATA / "members.csv", dtype=id_cols)
contrib = pd.read_csv(DATA / "contributions_monthly.csv", dtype=id_cols)
claims = pd.read_csv(DATA / "claims.csv", dtype=id_cols)

In [ ]:
# .shape gives (number of rows, number of columns).
print("employers:    ", employers.shape)
print("members:      ", members.shape)
print("contributions:", contrib.shape)
print("claims:       ", claims.shape)

In [ ]:
# How much memory does each table use? memory_usage(deep=True) counts the text too. 1e6 bytes = 1 MB.
print("employers:     ", round(employers.memory_usage(deep=True).sum() / 1e6, 1), "MB")
print("members:       ", round(members.memory_usage(deep=True).sum() / 1e6, 1), "MB")
print("contributions: ", round(contrib.memory_usage(deep=True).sum() / 1e6, 1), "MB")
print("claims:        ", round(claims.memory_usage(deep=True).sum() / 1e6, 1), "MB")

### The data dictionary

| Table | Column | Meaning |
|---|---|---|
| employers | `employer_no` | Employer registration number |
| | `sector`, `region`, `legal_form` | What the employer does and where |
| | `registration_date`, `status`, `closure_date` | When it registered; whether it's still trading |
| members | `member_no` | Member number, for life |
| | `employer_no` | Current or last employer |
| | `employment_start`, `registration_date` | When the job started; when the member was registered with the Fund |
| | `contribution_months` | Months of contributions on record, including before 2017 |
| | `exit_date`, `exit_reason`, `date_of_death`, `status` | How and when the member left, if they did |
| | `monthly_wage_tzs` | Latest gross monthly wage |
| contributions | `period` | The wage month the contribution is for (`YYYY-MM`) |
| | `members_reported`, `gross_wages_tzs` | From the employer's contribution schedule |
| | `contribution_due_tzs` | 20% of gross wages |
| | `amount_paid_tzs`, `payment_date` | What arrived and when. Empty means not paid by the snapshot date. |
| | `due_date` | Last day of the month after `period` |
| claims | `claim_no`, `member_no`, `benefit_type` | Which claim, whose, for what |
| | `claim_date`, `decision_date`, `status` | When it was lodged and decided; PAID, REJECTED or PENDING |
| | `amount_claimed_tzs`, `amount_paid_tzs` | Amounts. For pensions, the claim amount is the lump sum or first-year amount. |
| | `documents_missing_at_lodging`, `channel`, `branch` | How the claim came in |

### Theory: profiling a table in three commands

**Profiling** means getting a quick picture of a table before you analyse it. Three pandas commands do most of the
work:

- `head()` shows the first five rows. It answers *"what does one row look like?"*
- `info()` lists every column with its type and the number of values that are **not** missing. If a table has 13,800
  rows and a column shows 12,100 non-null values, 1,700 are missing.
- `describe()` summarises each number column with eight statistics.

The `describe()` statistics, in plain words, using a column of claim amounts as the example:

| Statistic | Meaning |
|---|---|
| `count` | how many values are filled in |
| `mean` | the average: add them up and divide by the count |
| `std` | standard deviation: roughly, how far a typical value is from the mean. Small = values bunched together |
| `min`, `max` | the smallest and largest values |
| `25%` | a quarter of the values are below this (the first **quartile**) |
| `50%` | half are below, half above: the **median**, the middle value |
| `75%` | three quarters are below this |

**A trick for spotting errors.** Compare `max` with `75%`. In clean data the largest value is usually a few times the
75th percentile. If it's a hundred times bigger, either something extreme happened or someone typed too many zeros.
`describe()` can't tell you which; it only tells you where to look.

## 4. First look

Start with the claims table. `head()` shows you what a row looks like. `info()` shows types and non-null
counts. `describe()` summarises the numbers.

In [ ]:
# The first five rows.
claims.head()

In [ ]:
# Column names, their types, and how many values are not missing.
claims.info()

In [ ]:
# Count, mean, standard deviation, min, quartiles and max of every numeric column.
claims.describe()

Look at `amount_claimed_tzs` in that summary. The maximum is far above the 75th percentile. That's either
a genuinely large claim or a typing error. Keep it in mind; you'll come back to it in Section 8.

**Your turn (1).** Run `head()`, `info()` and `describe()` on `contrib` and `members`. For each table, note
one thing that surprises you. Write it down, because you'll use these notes for the issues log at the end.

In [ ]:
# Your code here

> **Where we are.** You've loaded the tables and taken a first look, and you've started noting things that surprise
> you. Before you count or add up anything, you need to be sure what one row means.

### Theory: grain, keys and duplicates

**Grain** is the word for *"what one row represents"*. In `claims` the grain is one claim. In `contributions` it's one
employer in one month. Every question you ask of a table depends on the grain. "How many members do we have?" is
`len(members)` only if each member appears exactly once.

A **key** is the column (or set of columns) that identifies one row. If the grain is "one claim", the key is
`claim_no`, and no two rows should share a `claim_no`. When the grain needs two columns, as in "one employer, one
month", the key is both together, `employer_no` + `period`. That's called a **composite key**.

**Why duplicates are dangerous.** Imagine an employer whose March remittance of TZS 4 million appears twice by mistake.
Any total that includes March is now TZS 4 million too high. Nothing errors and nothing warns you: the sum just comes
out wrong, and it goes into a report.

There are two kinds of duplicate, and they need different treatment:

| Kind | What it looks like | Usual cause |
|---|---|---|
| **Exact duplicate** | every column identical | the same record loaded twice |
| **Key duplicate** | same key, but some columns differ | two versions of the truth: an update, a correction, two source systems |

An exact duplicate is safe to remove. A key duplicate isn't: you have to decide which version is right, and that
decision is a business rule, not a coding trick. That's why the next cells look at each case before changing anything.

## 5. What is one row? Keys and duplicates

Before you aggregate anything, you must know the **grain** of each table: what one row represents. If
you get this wrong, every sum you compute is wrong.

| Table | Expected grain | Key |
|---|---|---|
| employers | one employer | `employer_no` |
| members | one member | `member_no` |
| contributions | one employer, one month | `employer_no` + `period` |
| claims | one claim | `claim_no` |

Check whether the data agrees. For each table you'll ask three questions:

1. How many rows are there, and how many unique keys?
2. How many rows share their key with another row?
3. How many rows are exact copies of another row (every column identical)?

In [ ]:
# Members: one row per member_no?
print("rows:                        ", len(members))
print("unique member_no:            ", members["member_no"].nunique())

# duplicated(keep=False) marks EVERY row whose member_no appears more than once.
print("rows sharing a member_no:    ", members.duplicated(subset=["member_no"], keep=False).sum())

# duplicated() with no subset compares all columns.
print("exact duplicate rows:        ", members.duplicated(keep=False).sum())

In [ ]:
# Employers: one row per employer_no?
print("rows:                        ", len(employers))
print("unique employer_no:          ", employers["employer_no"].nunique())
print("rows sharing an employer_no: ", employers.duplicated(subset=["employer_no"], keep=False).sum())
print("exact duplicate rows:        ", employers.duplicated(keep=False).sum())

In [ ]:
# Contributions: one row per employer per month? The key is two columns together.
print("rows:                          ", len(contrib))
print("rows sharing employer + period:", contrib.duplicated(subset=["employer_no", "period"], keep=False).sum())
print("exact duplicate rows:          ", contrib.duplicated(keep=False).sum())

In [ ]:
# Claims: one row per claim_no?
print("rows:                ", len(claims))
print("unique claim_no:     ", claims["claim_no"].nunique())
print("rows sharing a key:  ", claims.duplicated(subset=["claim_no"], keep=False).sum())

None of the tables is clean. Look at each case before deciding what to do, because the right fix depends
on *why* the duplicate exists.

In [ ]:
# Keep only the members whose member_no appears more than once, and sort so the pairs sit together.
is_dup_member = members.duplicated(subset=["member_no"], keep=False)
dup_members = members[is_dup_member]
dup_members = dup_members.sort_values("member_no")

# Show a few useful columns.
dup_members[["member_no", "first_name", "surname", "phone", "region", "employer_no"]].head(6)

The duplicated members are the same person with the phone number written differently (`+255...` vs
`0...`). Probably the record was loaded twice from two systems. You can keep one row per `member_no`,
but first you need a standard phone format, or you'll keep whichever version comes first by accident.

Now the contributions:

In [ ]:
# How many payments are negative?
negative = contrib["amount_paid_tzs"] < 0
print("Negative amounts:", negative.sum())

In [ ]:
# Show the employer-months that have more than one row, sorted so the rows of one month sit together.
is_dup_month = contrib.duplicated(subset=["employer_no", "period"], keep=False)
dup_months = contrib[is_dup_month]
dup_months = dup_months.sort_values(["employer_no", "period", "amount_paid_tzs"])
dup_months.head(9)

There are two different stories in there:

1. **Exact duplicates.** The same row appears twice. That's an extraction problem, and one copy should go.
2. **A payment, the same payment again, and a negative amount.** That's a double posting that the
   finance team reversed. All three rows are genuine accounting entries, and they net to the right
   amount.

If you simply drop exact duplicates, you'll delete the second posting but keep the reversal, and the
employer will look like it paid nothing that month. The order in which you clean things matters. You'll
write that rule into the pipeline in Lab 3.

**Your turn (2).** The claims table has a unique `claim_no` on every row, so the key check passed. But can
the same member have lodged the same benefit twice within a few days? Find pairs of claims with the same
`member_no` and `benefit_type` lodged less than 10 days apart. How many are there? (Hint: sort, then
`groupby(...)["claim_date"].diff()`. You'll need the dates parsed first, so run Section 6 and come back to
this.)

In [ ]:
# Your code here

> **Where we are.** You've found duplicates in every table and understood the stories behind them. You haven't changed
> any data yet; you've only written down what you'll do. Next you'll turn text columns into proper dates and tidy
> categories, because until you do, you can't compare dates or group by region correctly.

### Theory: dates, "coercion", and valid versus plausible

**Why dates need converting.** After reading the CSV, a date like `"2024-03-15"` is just text. Text compares letter by
letter, so `"2024-3-5" > "2024-12-01"` is `True`, which is nonsense. Once converted to a real date (`datetime64`),
pandas understands that March comes before December, can subtract two dates to get days, and can pull out the year or
month with `.dt.year` and `.dt.month`.

**Coercion.** `pd.to_datetime(..., errors="coerce")` means: *"convert what you can; anything you can't read, turn into
`NaT` (Not a Time, the date version of missing)"*. Without `errors="coerce"`, one bad value like `"31/02/2024"` would
stop the whole conversion with an error. With it, the bad value quietly becomes missing. "Quietly" is the danger, so
you'll count how many values were filled in before and after converting. The difference is the number of unreadable
dates.

**Valid is not the same as plausible.** A date can be perfectly readable and still impossible: a payment in the year
2205, or a member registered at the age of 4. Each type of column has its own common-sense checks:

- dates shouldn't be after the day the extract was taken (31 January 2026, called `SNAPSHOT` in the code);
- a person's age at registration should be a working age;
- an amount paid shouldn't be negative unless it's a reversal.

**Categories need one spelling.** A computer treats `"DSM"`, `"Dar es salaam"` and `"Dar es Salaam "` (with a space
at the end) as three different regions. Any count by region splits Dar es Salaam into several rows. Making the spelling
consistent is called **standardising** a column.

## 6. Get the types right

### Dates

Parse the dates with `errors="coerce"`, which turns anything unparseable into `NaT` (not a time). Then
count how many values *were* present but didn't survive. Those are the broken ones.

Start with one column so you can see each step:

In [ ]:
# Step 1: how many birth dates are filled in before parsing?
filled_before = members["date_of_birth"].notna().sum()

# Step 2: convert the text to real dates. Anything that can't be read becomes NaT.
members["date_of_birth"] = pd.to_datetime(members["date_of_birth"], errors="coerce")

# Step 3: how many are filled in now? The difference is the number that couldn't be read.
filled_after = members["date_of_birth"].notna().sum()
print("date_of_birth values that could not be parsed:", filled_before - filled_after)

That's three lines for one column, and there are thirteen date columns. This is the right moment to write a small
**function**: the same steps, with the table and the column names as inputs.

In [ ]:
def parse_dates(df, columns):
    """Convert each column in `columns` from text to dates, and report any values that could not be read."""
    for col in columns:
        filled_before = df[col].notna().sum()
        df[col] = pd.to_datetime(df[col], errors="coerce")
        filled_after = df[col].notna().sum()
        if filled_before != filled_after:
            print(f"  {col}: {filled_before - filled_after} values could not be parsed")
    return df

In [ ]:
# Use the function on every table.
employers = parse_dates(employers, ["registration_date", "closure_date"])
members = parse_dates(members, ["employment_start", "registration_date", "exit_date", "date_of_death"])
contrib = parse_dates(contrib, ["due_date", "payment_date"])
claims = parse_dates(claims, ["claim_date", "decision_date"])
print("done")

Everything parsed. But parsing successfully doesn't make a date correct. The extract was taken on
31 January 2026, so any date after that is impossible. Check each date column against the snapshot:

In [ ]:
# Count dates after the snapshot, one column at a time.
print("employers.registration_date:", (employers["registration_date"] > SNAPSHOT).sum())
print("members.date_of_birth:      ", (members["date_of_birth"] > SNAPSHOT).sum())
print("members.registration_date:  ", (members["registration_date"] > SNAPSHOT).sum())
print("contrib.payment_date:       ", (contrib["payment_date"] > SNAPSHOT).sum())
print("claims.claim_date:          ", (claims["claim_date"] > SNAPSHOT).sum())
print("claims.decision_date:       ", (claims["decision_date"] > SNAPSHOT).sum())

In [ ]:
# Look at the impossible payment dates.
future_payment = contrib["payment_date"] > SNAPSHOT
contrib.loc[future_payment, ["employer_no", "period", "due_date", "payment_date", "amount_paid_tzs"]]

Payment dates in the 2200s. Someone typed `2205` for `2025`. You can't be certain what the right date was,
so in the pipeline you'll set these to missing and flag them rather than guessing.

Birth dates need a sanity check too. Nobody is registered with the Fund at age 10.

In [ ]:
# Age at registration, in years: (registration date - birth date) gives days; divide by 365.25.
days_old_at_reg = (members["registration_date"] - members["date_of_birth"]).dt.days
age_at_reg = days_old_at_reg / 365.25

age_at_reg.describe()

In [ ]:
# Show members who were apparently under 15 when they registered.
too_young = age_at_reg < 15
print(too_young.sum(), "members")
members.loc[too_young, ["member_no", "date_of_birth", "registration_date"]].head()

Those birth years look exactly 30 years too late (`2015` where `1985` was meant). A plausible typing slip,
but again not something to fix silently. Flag and set to missing.

### Categories

Region is supposed to be one of the mainland regions. Look at what's actually in the column:

In [ ]:
# value_counts() counts how many rows have each value.
members["region"].value_counts().head(25)

`DSM`, `Dar es salaam`, `Dar-es-Salaam` and `dar es salaam ` (with a trailing space) are all Dar es Salaam.
A report grouped by this column would show five different Dar es Salaams. Fix it one step at a time:

In [ ]:
# Step 1: remove spaces at the start and end of each value.
region = members["region"].str.strip()

# Step 2: make everything lower case, so "DSM" and "dsm" look the same.
region_lower = region.str.lower()

# Step 3: a dictionary of the spellings we want to correct, and what they should become.
REGION_FIX = {
    "dsm": "Dar es Salaam",
    "dar es salaam": "Dar es Salaam",
    "dar-es-salaam": "Dar es Salaam",
}

# Step 4: .replace() swaps any value found in the dictionary and leaves the rest alone.
region_fixed = region_lower.replace(REGION_FIX)
region_fixed.value_counts().head(8)

In [ ]:
# Step 5: everything else is now lower case ("arusha"). str.title() makes it "Arusha".
# But title() would turn "Dar es Salaam" into "Dar Es Salaam", so only apply it to the other regions.
is_dar = region_fixed == "Dar es Salaam"
region_fixed[~is_dar] = region_fixed[~is_dar].str.title()

# Step 6: save the cleaned column back into the table.
members["region"] = region_fixed
members["region"].value_counts().head(8)

That worked for members. The employers table has the same problem, and you'll clean regions again in Lab 3, so now
the steps are worth wrapping in a function:

In [ ]:
def clean_region(region_column):
    """Trim spaces, fix the Dar es Salaam spellings, and title-case every other region."""
    region = region_column.str.strip().str.lower()
    region = region.replace(REGION_FIX)
    is_dar = region == "Dar es Salaam"
    region[~is_dar] = region[~is_dar].str.title()
    return region


employers["region"] = clean_region(employers["region"])
employers["region"].value_counts().head(8)

**Your turn (3).** Do the same for `members.gender` (you'll find `F`, `M`, `Female`, `male` and missing values)
and for `claims.benefit_type` (some rows say `Maternity` or `Old Age Pension` instead of `MATERNITY`,
`OLD_AGE_PENSION`). Make both consistent and convert them to `category`.

In [ ]:
# Your code here

Here's one way to do it, so the rest of the notebook works whatever you wrote:

In [ ]:
# Gender: trim, upper case, then keep only the first letter: "Female" -> "F", "male" -> "M".
gender = members["gender"].str.strip().str.upper()
gender = gender.str[0]

# Anything that isn't F or M becomes missing (pd.NA).
gender[~gender.isin(["F", "M"])] = pd.NA

members["gender"] = gender.astype("category")
members["gender"].value_counts(dropna=False)

In [ ]:
# Benefit type: trim, upper case, and replace spaces with underscores: "Old Age Pension" -> "OLD_AGE_PENSION".
benefit = claims["benefit_type"].str.strip().str.upper()
benefit = benefit.str.replace(" ", "_")

claims["benefit_type"] = benefit.astype("category")
claims["benefit_type"].value_counts()

### What `.astype("category")` actually does

The `region` column has about 61,000 rows but only about 20 distinct values. Stored as ordinary text, pandas keeps a
string for every row:

```
"Dar es Salaam", "Arusha", "Dar es Salaam", "Mwanza", "Dar es Salaam", ...
```

Stored as a **category**, it keeps the list of distinct values once, and each row holds only a small number pointing
into that list:

```
categories:  0 → Arusha   1 → Dar es Salaam   2 → Mwanza   ...
column:      1, 0, 1, 2, 1, ...
```

The values you see when you print the column are exactly the same. Only the storage underneath changes.

### Memory

Categories aren't only tidier, they're smaller. Convert `region` and compare the memory before and after:

In [ ]:
# Memory used by region as text, in MB.
region_as_text_mb = members["region"].memory_usage(deep=True) / 1e6

# Convert to category, and measure again.
members["region"] = members["region"].astype("category")
employers["region"] = employers["region"].astype("category")
region_as_category_mb = members["region"].memory_usage(deep=True) / 1e6

print(f"region as text:     {region_as_text_mb:.2f} MB")
print(f"region as category: {region_as_category_mb:.2f} MB")

> **Where we are.** Dates are now real dates, and region, gender and benefit type each have one spelling. The next
> question is what to do about empty cells.

### Theory: the three reasons a value can be missing

Beginners often reach straight for "fill the gaps with the average". That can be exactly the wrong thing to do. First
ask **why** the value is missing. There are three common reasons:

1. **It doesn't apply.** `date_of_death` is empty because the member is alive. The empty cell *is* the information.
   Filling it in would kill thousands of members on paper.
2. **It's missing by chance.** A clerk skipped a field now and then, with no pattern. Filling in a typical value is
   usually harmless here.
3. **It's missing for a reason.** The value is missing more often for some groups than others: older records, one
   branch, one source system. The pattern itself is a finding. Filling the gaps would hide it, and any model trained on
   the filled data would quietly learn the pattern instead.

The way to tell them apart is to **look at the missing values against other columns**: by year, by region, by channel.
If the share missing is flat, it's probably chance. If it jumps for one group, there's a reason, and someone in the
business can usually tell you what it is.

On 61,000 rows it doesn't matter much. On the full contribution history (millions of rows, Day 3) it's
the difference between fitting in memory and not.

## 7. Missing values: what does "missing" mean here?

Count them first:

In [ ]:
# isna() marks missing values as True; .sum() counts the Trues in each column.
missing_count = members.isna().sum()

# The same as a share of all rows (mean of True/False = share of True).
missing_share = members.isna().mean()

# Put both in one table, keep only columns that have something missing, largest first.
missing_table = pd.DataFrame({"missing": missing_count, "share": missing_share})
missing_table = missing_table[missing_table["missing"] > 0]
missing_table.sort_values("share", ascending=False)

Now the important part. For each column, ask **why** it's missing. The answers are very different:

- `date_of_death` is missing for almost everyone because almost everyone is alive. That's correct. Filling it in
  would be a serious mistake.
- `exit_date` and `exit_reason` are missing for active members. Also correct.
- `nida_no` is missing for about 1 in 10. Is that random, or is there a pattern?

In [ ]:
# Add a column with the year each member registered.
members["reg_year"] = members["registration_date"].dt.year

# A True/False column: is the NIDA number missing?
members["nida_missing"] = members["nida_no"].isna()

# For each registration year, the share of members with no NIDA number (mean of True/False).
nida_gap = members.groupby("reg_year")["nida_missing"].mean()
nida_gap

In [ ]:
# Draw it as a bar chart.
nida_gap.plot(kind="bar", color="#2a78d6")
plt.title("Share of members with no NIDA number, by registration year")
plt.ylabel("share missing")
plt.xlabel("registration year")
plt.show()

Missing NIDA (National Identification Authority) numbers are concentrated among members registered before 2019. That fits with NIDA numbers
becoming a registration requirement later. It isn't random, so it's a **task** for the member
services team (update older records), not something to impute.

Same exercise for contributions:

In [ ]:
# Missing values in the contributions table.
contrib_missing = pd.DataFrame({"missing": contrib.isna().sum(), "share": contrib.isna().mean()})
contrib_missing[contrib_missing["missing"] > 0]

`amount_paid_tzs` and `payment_date` are missing together on the same rows. Those are months the
employer hasn't paid. In business terms they're **arrears**, and they're one of the most useful things in
this whole extract. `gross_wages_tzs` missing on a few rows is a different matter: the employer
submitted a schedule without wages. That's a genuine data gap.

In [ ]:
# How many employer-months are unpaid?
unpaid = contrib["amount_paid_tzs"].isna()
print("Unpaid employer-months:", unpaid.sum())
print("Share of all employer-months:", round(unpaid.mean() * 100, 1), "%")

# Are amount and date always missing on the same rows?
no_date = contrib["payment_date"].isna()
print("Rows where only one of the two is missing:", (unpaid != no_date).sum())

> **Where we are.** You know which empty cells are real gaps and which are information. Now you'll look at the values
> that *are* there, one column at a time, to see their shape and spot the extremes.

### Theory: distributions, skew, and outliers

**A distribution** is the pattern of values in a column: where most of them sit, how spread out they are, and how far
the extremes reach. A **histogram** draws it: the values are cut into ranges ("bins"), and each bar shows how many fall
in that range.

**Skewed data.** Wages aren't spread evenly around an average. Most people earn a moderate amount and a few earn a lot,
so the histogram has a big hump on the left and a long thin tail to the right. That's called **right-skewed**. Money
amounts are almost always like this: wages, claim amounts, contributions.

**Why the median beats the mean for skewed data.** Take five monthly wages: 400k, 500k, 600k, 700k and 12 million.
The mean is 2.84 million, which describes nobody. The median (the middle value) is 600k, which describes a typical
member. One extreme value moves the mean a lot and the median not at all.

**The log scale.** A logarithm (base 10) counts the zeros: log10(1,000) = 3, log10(1,000,000) = 6. Plotting wages on a
log scale turns "times ten" into "plus one", so TZS 100,000, 1 million and 10 million sit at evenly spaced points. The
long tail shrinks and the shape of the hump becomes visible.

**Outlier or error?** An **outlier** is a value far from the rest. It could be real (a managing director's salary) or
an error (an extra zero). The data alone can't settle it. What helps is **context**: compare the value with similar
rows, such as other staff at the same employer. A wage ten times the employer's median, at an exact multiple of ten,
looks like a typing slip. Flag it for a person to check; don't "correct" it in code.

## 8. One variable at a time

### Wages

In [ ]:
# Drop the missing wages, then draw a histogram (how many members fall in each wage range).
wages = members["monthly_wage_tzs"].dropna()

plt.hist(wages, bins=80, color="#2a78d6")
plt.title("Monthly wage (TZS)")
plt.xlabel("TZS")
plt.ylabel("members")
plt.show()

In [ ]:
# The same data on a log scale: log10(1,000,000) = 6, log10(10,000,000) = 7.
# Each step of 1 on this axis means "ten times more".
plt.hist(np.log10(wages), bins=80, color="#2a78d6")
plt.title("log10 of monthly wage")
plt.xlabel("log10(TZS)")
plt.ylabel("members")
plt.show()

In [ ]:
# The 1st percentile, median (50%) and 99th percentile tell you more than the mean for skewed data.
wages.describe(percentiles=[0.01, 0.5, 0.99])

The raw histogram is almost useless: a few huge values squash everything else into one bar. On a log
scale you can see the real shape, a skewed distribution centred somewhere around TZS 600,000-800,000.
Wages are almost always skewed like this, which is why medians describe them better than means.

What about the extreme values?

In [ ]:
# The 8 highest wages.
members.nlargest(8, "monthly_wage_tzs")[["member_no", "monthly_wage_tzs", "employer_no"]]

Some of these end in an extra zero compared with the rest of their employer's staff. Compare each one
with the median wage at the same employer:

In [ ]:
# For every member, the median wage of everyone at the same employer.
# groupby(...).transform("median") returns one value per ROW (not per employer), so it lines up with members.
members["employer_median_wage"] = members.groupby("employer_no")["monthly_wage_tzs"].transform("median")

# How many times the employer median is each member's wage?
members["wage_vs_employer"] = members["monthly_wage_tzs"] / members["employer_median_wage"]

# Show members earning 8 or more times their employer's median.
suspect = members[members["wage_vs_employer"] > 8]
suspect[["member_no", "employer_no", "monthly_wage_tzs", "employer_median_wage"]]

A wage eight or more times the employer's median, at round multiples of ten, is very likely a typing
error. It could also be a managing director. That's the kind of call you **flag for someone to check**,
not something you "fix" in code.

### Ages

In [ ]:
# Age at the snapshot date, in years.
members["age"] = (SNAPSHOT - members["date_of_birth"]).dt.days / 365.25

# Only active members.
active_members = members[members["status"] == "ACTIVE"]

plt.hist(active_members["age"].dropna(), bins=45, color="#2a78d6")
plt.title("Age of active members")
plt.xlabel("age at snapshot")
plt.ylabel("members")
plt.show()

### Claim amounts by benefit type

A single histogram of all claim amounts would mix TZS 500,000 funeral grants with multi-million pension
lump sums. Look at each benefit type separately:

In [ ]:
# describe() for each benefit type, keeping four of its columns.
amount_by_benefit = claims.groupby("benefit_type", observed=True)["amount_claimed_tzs"].describe()
amount_by_benefit[["count", "min", "50%", "max"]]

**Your turn (4).** The funeral grant is a flat TZS 500,000. Which funeral grant claims have a different
amount? What do you think happened?

In [ ]:
# Your code here

> **Where we are.** You've looked at the number columns one by one: wages are skewed with a few likely typing slips,
> ages and claim amounts have their own checks. Now you'll count the text columns (sector, region) and start grouping:
> the step from "what's in this column" to "how do groups compare".

### Theory: group-by in one picture

`groupby` is the most useful tool in pandas. It does three things, often called **split, apply, combine**:

```
split            apply (e.g. mean)       combine
sector=A: rows ─► mean of A ─┐
sector=B: rows ─► mean of B ─┼─► one small table: one row per sector
sector=C: rows ─► mean of C ─┘
```

`c.groupby("employer_no")["contribution_due_tzs"].mean()` means: split the contribution rows by employer, take the mean
amount due for each employer, and put the results together in one table with one row per employer.

`.agg(...)` lets you compute several summaries at once (the count of months, the average due, the average headcount),
each becoming one column of the result.

The grain changes when you group. The input had one row per employer per month; the output has one row per employer.
Always know which grain you're looking at.

## 9. Categories and groups

In [ ]:
# Number of employers in each sector, as a horizontal bar chart.
sector_counts = employers["sector"].value_counts().sort_values()

sector_counts.plot(kind="barh", color="#2a78d6")
plt.title("Employers by sector")
plt.xlabel("employers")
plt.show()

In [ ]:
# Number of members in each region, top 12.
region_counts = members["region"].value_counts().head(12).sort_values()

region_counts.plot(kind="barh", color="#2a78d6")
plt.title("Members by region (top 12)")
plt.xlabel("members")
plt.show()

### Contributions per employer per month

Since the contributions table has one row per employer per month (once duplicates are dealt with), an
employer's typical monthly contribution is a simple group-by. For now, drop exact duplicates and the
reversal rows so that you're not double counting. You'll do this properly in Lab 3.

In [ ]:
# A quick, rough clean-up for exploring (Lab 3 does this properly):
# 1. drop rows that are exact copies
c = contrib.drop_duplicates()

# 2. drop the negative reversal rows
c = c[c["amount_paid_tzs"].isna() | (c["amount_paid_tzs"] >= 0)]

# 3. keep one row per employer-month
c = c.drop_duplicates(subset=["employer_no", "period"])
print("rows left:", len(c))

In [ ]:
# For each employer: number of months, average contribution due, and average members reported.
per_employer = c.groupby("employer_no").agg(
    months=("period", "nunique"),
    avg_due=("contribution_due_tzs", "mean"),
    avg_members=("members_reported", "mean"),
)

# Largest contributors first.
per_employer.sort_values("avg_due", ascending=False).head(10)

### Claims per benefit type and region

In [ ]:
# Claims don't have a region, but members do. Keep one row per member, then join the region onto each claim.
members_one_row = members.drop_duplicates(subset=["member_no"])

claims_with_region = claims.merge(members_one_row[["member_no", "region"]], on="member_no", how="left")

# crosstab counts claims for every (region, benefit type) pair.
region_by_benefit = pd.crosstab(claims_with_region["region"], claims_with_region["benefit_type"])

# Add a total column, sort by it, and show the 10 busiest regions.
region_by_benefit["total"] = region_by_benefit.sum(axis=1)
region_by_benefit.sort_values("total", ascending=False).head(10)

### Theory: reading a time series

A **time series** is a set of values measured at regular points in time: here, one total per month. When you plot one,
look for four things:

- **Trend:** the long-run direction. Contributions due rise over the years as membership and wages grow.
- **Seasonality:** a pattern that repeats every year in the same months. At the Fund, more cash arrives in June
  because of the year-end compliance drive.
- **Steps:** a sudden permanent jump, usually caused by a rule change (a new wage floor, a new contribution rate).
- **Shocks:** a temporary dip or spike with an outside cause, such as COVID-19 in 2020.

**Two clocks in one table.** Each contribution row has a **wage month** (`period`, the month the wages were earned) and
a **payment date** (when the money arrived). Grouping by wage month tells you what was *owed* each month. Grouping by
payment month tells you what *arrived* each month. These are different series with different shapes, and Day 4's
forecasts need to know which one finance means.

**On-time rate.** A payment is on time if it arrived by the due date (the last day of the month after the wage month).
The on-time rate for a month is the share of employers whose payment for that month was on time.

## 10. Time: trends, seasons and late payment

Contributions are due by the last day of the month after the wage month. How much is due each month, and
how much of it arrives on time?

In [ ]:
# Turn the period text "2024-03" into a real date: the first day of that month.
c["period_start"] = pd.to_datetime(c["period"] + "-01")

# Days between the due date and the payment date. Negative = paid early.
c["days_late"] = (c["payment_date"] - c["due_date"]).dt.days

# Paid on time? (True / False)
c["on_time"] = c["days_late"] <= 0

c[["employer_no", "period", "due_date", "payment_date", "days_late", "on_time"]].head()

In [ ]:
# One row per month: total due, total paid, and the share of employers paying on time.
monthly = c.groupby("period_start").agg(
    due=("contribution_due_tzs", "sum"),
    paid=("amount_paid_tzs", "sum"),
    on_time=("on_time", "mean"),
)
monthly.head()

In [ ]:
# Plot due and paid, in billions of TZS.
plt.plot(monthly.index, monthly["due"] / 1e9, color="#b5b4ad", label="due")
plt.plot(monthly.index, monthly["paid"] / 1e9, color="#2a78d6", label="paid")
plt.title("Contributions due and paid, by wage month")
plt.ylabel("TZS billions")
plt.legend()
plt.show()

In [ ]:
# Plot the on-time share, and shade the COVID-19 months.
plt.plot(monthly.index, monthly["on_time"], color="#2a78d6")
plt.axvspan(pd.Timestamp("2020-03-01"), pd.Timestamp("2020-10-01"), color="#eb6834", alpha=0.15)
plt.text(pd.Timestamp("2020-04-01"), monthly["on_time"].min(), "COVID-19", color="#eb6834")
plt.title("Share of employer-months paid by the due date")
plt.show()

Two things stand out. The steady growth in the amount due (more members, higher wages, and a step in
January 2023) and a clear dip in on-time payment in 2020.

Now look at **when** payments actually arrive, by calendar month of the payment date:

In [ ]:
# Keep only rows that were paid, and add the calendar month (1-12) of the payment.
paid_rows = c[c["payment_date"].notna()].copy()
paid_rows["payment_month"] = paid_rows["payment_date"].dt.month

# Total paid in each calendar month, in billions.
paid_by_month = paid_rows.groupby("payment_month")["amount_paid_tzs"].sum() / 1e9

paid_by_month.plot(kind="bar", color="#2a78d6")
plt.title("Total paid by calendar month of payment (TZS billions)")
plt.xlabel("month of payment")
plt.show()

June stands out. The Fund's financial year ends on 30 June, and a compliance drive before year-end pulls in
old arrears. It matters for Day 4: if you forecast *cash received*, June behaves differently from other
months; if you forecast *contributions due*, it doesn't.

**Your turn (5).** Which sector has the worst on-time payment rate? Join `c` to `employers` on
`employer_no` and compute the on-time share by sector, from worst to best.

In [ ]:
# Your code here

> **Where we are.** So far you've looked at each table on its own. Many of the most useful questions need two tables
> together: "which regions lodge the most claims?" needs claims (no region) and members (with region). That means
> joining.

### Theory: how a join works, and how it goes wrong

A **join** (pandas calls it `merge`) lines up rows from two tables that share a key. For every claim, pandas looks up
the member with the same `member_no` and copies that member's columns onto the claim row.

**Left join.** `how="left"` keeps every row of the left table (claims), whether or not it finds a match. If a claim's
member isn't found, the member columns are left empty. That's what you want here: you mustn't lose claims just because
a member record is missing.

**Three ways a join goes wrong**, all silently:

| Problem | What happens | How to catch it |
|---|---|---|
| **Duplicate keys on the right** | a claim matches two member rows and appears twice | compare row counts before and after |
| **No match** | the claim keeps empty member columns | `indicator=True` adds a `_merge` column saying `both` or `left_only` |
| **Clashing column names** | both tables have `status`, pandas renames them `status_x` and `status_y` | rename before joining |

The rule: **count the rows before and after every join, and look at the rows that didn't match.** The unmatched rows
are often the most interesting finding of the day.

## 11. Joining tables, and checking the joins

Every join is a place where rows can quietly appear or disappear. Always compare row counts before and
after, and look at what didn't match.

In [ ]:
# The member columns we want on each claim. Both tables have a "status" column, so rename the member one.
member_cols = members_one_row[["member_no", "gender", "date_of_birth", "date_of_death", "status"]]
member_cols = member_cols.rename(columns={"status": "member_status"})

# Join. how="left" keeps every claim. indicator=True adds a "_merge" column saying where each row matched.
rows_before = len(claims)
cm = claims.merge(member_cols, on="member_no", how="left", indicator=True)

print("rows before:", rows_before)
print("rows after: ", len(cm))
cm["_merge"].value_counts()

Both tables have a `status` column (claim status and member status), so the member one is renamed before
the join. Otherwise pandas would call them `status_x` and `status_y`, and nobody would remember which is
which.

The row count didn't change, which means `member_no` is unique on the right-hand side. That's why you
dropped the duplicate members first. If you hadn't, every claim by a duplicated member would have
been counted twice.

But some claims have no matching member at all:

In [ ]:
# "left_only" = the claim found no member.
orphans = cm[cm["_merge"] == "left_only"]
print(len(orphans), "claims for member numbers that don't exist in the members table")
orphans[["claim_no", "member_no", "benefit_type", "claim_date", "amount_paid_tzs", "status"]].head()

### Theory: correlation in plain words

**Correlation** is one number between -1 and +1 that says how closely two columns move together:

| Value | Meaning | Example |
|---|---|---|
| close to +1 | when one is high, the other is high | headcount and total wages |
| close to 0 | no straight-line relationship | shoe size and on-time payment |
| close to -1 | when one is high, the other is low | days late and on-time share |

Three cautions:

- **Correlation isn't cause.** Large employers might pay on time because they have finance departments, or because
  the Fund chases them harder. The number can't tell you which.
- **It only sees straight lines.** A relationship that goes up and then down can have a correlation near 0.
- **Skewed columns distort it.** A few huge employers can dominate. Taking the logarithm of headcount first (as the
  code does) gives the many small employers a fair say.

Paid claims for members who don't exist are a serious finding. It might be a migration gap between
systems, or it might be something worse. Either way, that list goes to the claims team.

## 12. Relationships between variables

Do small employers pay late more often than large ones? Band employers by average headcount and compare:

In [ ]:
# One row per employer: average headcount and share of months paid on time.
emp_stats = c.groupby("employer_no").agg(
    avg_members=("members_reported", "mean"),
    on_time=("on_time", "mean"),
)

# pd.cut puts each employer into a size band.
emp_stats["size_band"] = pd.cut(
    emp_stats["avg_members"],
    bins=[0, 5, 20, 100, 10_000],
    labels=["1-5", "6-20", "21-100", "100+"],
)

# Average on-time share, and number of employers, per band.
emp_stats.groupby("size_band", observed=True)["on_time"].agg(["mean", "count"])

In [ ]:
# Correlation: +1 = move together perfectly, 0 = no relationship, -1 = opposite directions.
# Headcount is very skewed, so use its log. clip(lower=1) avoids log(0).
log_headcount = np.log(emp_stats["avg_members"].clip(lower=1))
correlation = log_headcount.corr(emp_stats["on_time"])
print("Correlation between log(headcount) and on-time share:", round(correlation, 3))

> **Where we are.** You've profiled, de-duplicated, typed, grouped, joined and correlated. The data is now good enough
> to ask business questions of it. This section looks for patterns that may point to real problems: claims and
> payments that a person in the business should look at.

### Theory: a rule, a flag, and a decision

The checks in this section are **rules**: simple tests like "claim lodged after the member's recorded death". A rule
produces a **flag**: a list of rows that match. A flag is *not* a finding of wrongdoing. Each flagged row needs a
person who knows the business to make the **decision**.

A good rule balances two kinds of mistake:

- **Too broad**, and it flags thousands of legitimate rows (every funeral grant comes after a death, by design). The
  team stops trusting the list.
- **Too narrow**, and it misses real cases.

Getting that balance right is the heart of Day 2's risk model and Day 5's fraud detection. Here you're doing it by
hand, with rules you can read.

The relationship is weak. Size alone doesn't tell you much about payment behaviour. That's a useful
result: it tells you a compliance model on Day 2 will need more than headcount.

## 13. Signals worth a closer look

These aren't conclusions. They're patterns that someone who knows the business should look at.

**Claims lodged after the member's recorded death**, for benefits that the member would have to claim
personally:

In [ ]:
# Benefits a member has to claim in person.
personal = ["MATERNITY", "UNEMPLOYMENT", "OLD_AGE_PENSION", "OLD_AGE_GRANT", "INVALIDITY_PENSION"]

# Three conditions, each a True/False column.
has_death_date = cm["date_of_death"].notna()
claimed_after_death = cm["claim_date"] > cm["date_of_death"]
is_personal = cm["benefit_type"].isin(personal)

# & means "and": all three must be True.
after_death = cm[has_death_date & claimed_after_death & is_personal]
print(len(after_death), "claims")
after_death[["claim_no", "member_no", "benefit_type", "date_of_death", "claim_date", "amount_paid_tzs", "branch"]].head(8)

Notice that survivors' pensions and funeral grants are excluded on purpose: those are *supposed* to come
after a death. A rule that flags every claim after a death would bury the real cases in false alarms.

**Maternity claims by members recorded as male:**

In [ ]:
is_maternity = cm["benefit_type"] == "MATERNITY"
is_male = cm["gender"] == "M"

maternity_male = cm[is_maternity & is_male]
maternity_male[["claim_no", "member_no", "claim_date", "amount_paid_tzs", "branch"]]

It could be a gender data error in the member record, or a claim against the wrong member number.
Either way, a person should look at it.

**Employers still trading but falling behind.** Count unpaid months in 2025 for employers recorded as
ACTIVE. Be careful with the most recent months: December's contributions were only due on 31 January,
the day of the extract, so "unpaid" there mostly means "not due yet". Only count months that were at least
30 days overdue when the extract was taken.

In [ ]:
# Months from January 2025 that were due at least 30 days before the snapshot.
cutoff = SNAPSHOT - pd.Timedelta(days=30)
recent = c[(c["period_start"] >= "2025-01-01") & (c["due_date"] <= cutoff)].copy()

# Mark the unpaid months, then count them per employer.
recent["unpaid"] = recent["amount_paid_tzs"].isna()
unpaid_per_employer = recent.groupby("employer_no")["unpaid"].sum()
unpaid_per_employer.head()

In [ ]:
# The employer numbers of ACTIVE employers.
active_employer_nos = employers.loc[employers["status"] == "ACTIVE", "employer_no"]

# Employers with 3 or more unpaid months...
three_or_more = unpaid_per_employer[unpaid_per_employer >= 3]

# ...that are still ACTIVE.
behind = three_or_more[three_or_more.index.isin(active_employer_nos)]
print(len(behind), "active employers with 3 or more overdue unpaid months in 2025")
behind.sort_values(ascending=False).head(8)

Each of those employers has staff whose statements are missing months, and some of those staff will
end up at a branch complaining (you'll meet their complaints on Day 6). A list like this is what a
compliance officer wants on a Monday morning.

> **Where we are.** You now know this data better than most people who use it. You've found duplicates, impossible
> dates, inconsistent spellings, meaningful gaps, probable typing errors, orphaned claims and payment patterns. All of
> that is still only in this notebook.

### Theory: why an issues log

An **issues log** is a simple table with one row per data problem: which table, which column, what's wrong, how many
rows, and what to do about it. It does three jobs:

1. **It's shareable.** The data owners (member services, finance, claims) can read it without opening Python.
2. **It's trackable.** Each issue can be given an owner and marked fixed.
3. **It becomes code.** In Lab 3, each "what to do" column turns into a cleaning rule in the pipeline. The log is the
   specification for that code.

## 14. Write it down: the issues log

Findings that live only in your head, or in a notebook nobody reads, get lost. Put them in a table that
you can share, track and turn into pipeline rules in Lab 3.

First work out the counts you'll quote, one per line, so you can check each number:

In [ ]:
n_dup_members = members.duplicated(subset=["member_no"]).sum()
n_bad_dob = too_young.sum()
n_wage_outliers = len(suspect)
n_no_nida = members["nida_no"].isna().sum()
n_dup_employers = employers.duplicated(subset=["employer_no"]).sum()
n_exact_dup_contrib = contrib.duplicated().sum()
n_negative = (contrib["amount_paid_tzs"] < 0).sum()
n_future_payment = (contrib["payment_date"] > SNAPSHOT).sum()
n_no_wages = contrib["gross_wages_tzs"].isna().sum()
n_orphans = len(orphans)
n_after_death = len(after_death)

is_funeral = claims["benefit_type"] == "FUNERAL_GRANT"
n_wrong_funeral = (is_funeral & (claims["amount_claimed_tzs"] != 500_000)).sum()

In [ ]:
# One list per issue: table, column, what's wrong, rows affected, what to do.
issues = pd.DataFrame(
    [
        ["members", "member_no", "Duplicate member rows differing only in phone format", n_dup_members, "Standardise phone to +255, keep one row per member_no"],
        ["members", "region", "Several spellings of Dar es Salaam; case and spacing variants", None, "Map to canonical region names"],
        ["members", "gender", "Mixed codes (F/M/Female/male) and missing", None, "Standardise to F/M; leave missing as missing"],
        ["members", "date_of_birth", "Birth dates giving age under 15 at registration", n_bad_dob, "Set to missing and flag for correction"],
        ["members", "monthly_wage_tzs", "Wages 8x+ the employer median, likely extra zero", n_wage_outliers, "Flag for review; don't auto-correct"],
        ["members", "nida_no", "Missing, mostly for pre-2019 registrations", n_no_nida, "Member services data-update task"],
        ["employers", "employer_no", "Duplicate employer rows with upper-case names", n_dup_employers, "Keep one row per employer_no"],
        ["contributions", "employer_no+period", "Exact duplicate rows", n_exact_dup_contrib, "Drop exact duplicates, but only where there's no reversal"],
        ["contributions", "amount_paid_tzs", "Negative amounts: reversals of double postings", n_negative, "Net payments per employer-month"],
        ["contributions", "payment_date", "Payment dates after the snapshot (typed 22xx)", n_future_payment, "Set to missing and flag"],
        ["contributions", "gross_wages_tzs", "Missing on some schedules", n_no_wages, "Derive from contribution_due / 0.2"],
        ["claims", "benefit_type", "Inconsistent labels", None, "Standardise to upper-case codes"],
        ["claims", "member_no", "Claims for member numbers not in members table", n_orphans, "Refer to claims team"],
        ["claims", "claim_date", "Personal benefit claims after recorded death", n_after_death, "Refer to claims/fraud team"],
        ["claims", "amount_claimed_tzs", "Funeral grants not equal to TZS 500,000", n_wrong_funeral, "Check against payment records"],
    ],
    columns=["table", "column", "issue", "rows_affected", "proposed_action"],
)
issues

In [ ]:
# Save the log as a CSV file in a "reports" folder (created if it doesn't exist).
Path("reports").mkdir(exist_ok=True)
issues.to_csv("reports/data_quality_issues_log.csv", index=False)
print("Saved reports/data_quality_issues_log.csv")

**Your turn (6).** Add at least two rows of your own from what you noticed in *Your turn (1)* and the rest
of the notebook.

In [ ]:
# Your code here

## 15. Questions to take back to the business

Good exploration ends with better questions, not only cleaner data. Here's a starting list. Add at least two of
your own, and keep the list: you'll come back to it on Day 10 when you prioritise what the Fund should build.

1. Are the 30-odd paid claims for unknown member numbers a migration problem or something else? Who owns
   that check today?
2. Should active employers with three or more unpaid months go straight to the compliance team's follow-up
   list? Who decides the threshold?
3. The June payment peak comes from the year-end compliance drive. Would finance prefer forecasts of
   contributions *due* or of cash *received*? (You'll need the answer on Day 4.)
4. What share of members registered before 2019 still lack a NIDA number, and is that blocking anything,
   for example digital life verification?
5. Which of the issues above should be fixed at the source system, rather than cleaned downstream every
   month?

---

## Answers to the "Your turn" exercises

Try each one yourself first. These are one way of doing it; yours may be different and still right.

**(1)** There's no single right answer. Things people usually notice: `amount_paid_tzs` has negative
values and a lot of missing values; `members_reported` has a minimum of 0; `contribution_months` goes into
the hundreds for long-serving members; `monthly_wage_tzs` has a very large maximum.

In [ ]:
contrib.describe()

**(2)** Near-duplicate claims:

In [ ]:
# Sort so each member's claims for the same benefit sit together, oldest first.
sorted_claims = claims.sort_values(["member_no", "benefit_type", "claim_date"])

# diff() = days since the previous claim in the same (member, benefit) group. The first one in a group is empty.
gap = sorted_claims.groupby(["member_no", "benefit_type"], observed=True)["claim_date"].diff()
sorted_claims["days_since_previous"] = gap.dt.days

near_dupes = sorted_claims[sorted_claims["days_since_previous"] < 10]
print(len(near_dupes), "claims lodged within 10 days of an earlier claim for the same benefit by the same member")
near_dupes[["claim_no", "member_no", "benefit_type", "claim_date", "amount_claimed_tzs", "status"]].head()

**(3)** See the cells right after the exercise.

**(4)** Funeral grants with the wrong amount:

In [ ]:
is_funeral = claims["benefit_type"] == "FUNERAL_GRANT"
wrong_amount = claims["amount_claimed_tzs"] != 500_000

claims.loc[is_funeral & wrong_amount, ["claim_no", "claim_date", "amount_claimed_tzs", "amount_paid_tzs", "status"]]

TZS 5,000,000 is 500,000 with an extra zero. If these were *paid* at that amount, it's an overpayment
to recover. That's a question for finance, and a good argument for a validation rule at data entry.

**(5)** On-time rate by sector:

In [ ]:
# Add each employer's sector to the contribution rows.
employers_one_row = employers.drop_duplicates(subset=["employer_no"])
c_with_sector = c.merge(employers_one_row[["employer_no", "sector"]], on="employer_no", how="left")

# Average on-time share per sector, worst first.
c_with_sector.groupby("sector")["on_time"].mean().sort_values()

**(6)** Your own rows, for example:

In [ ]:
n_zero_members = (contrib["members_reported"] == 0).sum()
n_maternity_male = len(maternity_male)

extra = pd.DataFrame(
    [
        ["contributions", "members_reported", "Zero members reported with non-zero wages", n_zero_members, "Query employer schedule"],
        ["claims", "gender vs benefit", "Maternity claims against male member records", n_maternity_male, "Check member record and claim"],
    ],
    columns=issues.columns,
)

# pd.concat stacks two tables on top of each other.
issues_extended = pd.concat([issues, extra], ignore_index=True)
issues_extended.tail(3)

## Looking back, looking ahead

**What you did in this lab.** You took four raw tables and, step by step:

1. learned how pandas stores data and why column types matter;
2. loaded the tables with IDs kept as text;
3. checked the grain of each table and found exact duplicates, key duplicates and reversals;
4. turned dates into real dates, found impossible ones, and standardised categories;
5. worked out *why* values were missing instead of filling them in;
6. looked at distributions and found probable typing errors in wages and claim amounts;
7. joined claims to members and found claims with no member at all;
8. wrote everything into an issues log and a list of questions for the business.

**What carries forward.** The issues log in `reports/` is the input to Lab 3. There you'll build a pipeline that
applies a fix for each issue in a fixed order, so that anyone can rerun it on next month's extract and get the same
clean tables.

**Where it leads.** The clean claims feed Day 2's risk score, the clean contributions feed Day 4's forecasts, and the
habits you practised here (check the grain, count before and after, ask why it's missing) are the ones you'll use on
every dataset for the rest of the course.